In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# ✅ Updated file path
file_path = '../data/processed/enriched_options_final.csv'
df = pd.read_csv(file_path)

# ✅ Check shape
print(f"✅ Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# ✅ Expected columns
expected_columns = [
    'contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change',
    'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney',
    'contractSize', 'currency', 'ticker', 'expiry', 'days_to_expiry', 'spot_price',
    'moneyness', 'date_collected', 'delta', 'gamma', 'theta', 'vega',
    'iv_smoothed_5d', 'log_return', 'hist_vol_20d', 'moneyness_class', 'label'
]

# ✅ Check if all expected columns are present
missing_cols = [col for col in expected_columns if col not in df.columns]
if missing_cols:
    print(f"❌ Missing columns: {missing_cols}")
else:
    print("✅ All expected columns are present.")

# 🧾 Preview the first few rows
df.head()


✅ Dataset Loaded: 1869 rows, 43 columns
✅ All expected columns are present.


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,hist_vol_20d,moneyness_class,day_of_week,month,volume_oi_ratio,bid_ask_spread,mid_price,last_mid_diff,vega,daily_sentiment_score
0,AAPL250703C00110000,2025-06-27 19:54:05+00:00,110.0,91.49,90.35,91.55,0.489998,0.538459,46.0,418,...,NaN,OTM,6,6,0.109785,1.20,90.950,0.540,0.002382,0
1,AAPL250703C00140000,2025-06-26 18:49:08+00:00,140.0,61.36,60.85,61.55,0.000000,0.000000,2.0,1913,...,NaN,OTM,6,6,0.001045,0.70,61.200,0.160,0.000926,0
2,AAPL250703C00145000,2025-06-20 16:42:17+00:00,145.0,53.74,55.85,56.60,0.000000,0.000000,11.0,70,...,NaN,OTM,6,6,0.154930,0.75,56.225,-2.485,0.001277,0
3,AAPL250703C00150000,2025-06-27 17:33:19+00:00,150.0,52.13,50.05,52.35,2.030003,4.051901,1.0,14,...,NaN,OTM,6,6,0.066667,2.30,51.200,0.930,0.001181,0
4,AAPL250703C00155000,2025-06-20 15:52:48+00:00,155.0,43.55,45.85,46.60,0.000000,0.000000,1.0,1,...,NaN,OTM,6,6,0.500000,0.75,46.225,-2.675,0.001637,0


In [2]:
# Advanced Feature Engineering

# Mid-price to avoid skew from lastPrice
df['mid_price'] = (df['bid'] + df['ask']) / 2

# Relative differences
df['bs_pct_diff'] = (df['bs_price'] - df['mid_price']) / df['mid_price']
df['mc_pct_diff'] = (df['mc_price'] - df['mid_price']) / df['mid_price']
df['norm_spread'] = (df['ask'] - df['bid']) / df['mid_price']

# Interaction terms
df['theta_days'] = df['theta'] * df['days_to_expiry']
df['delta_spot'] = df['delta'] * df['spot_price']
df['gamma_moneyness'] = df['gamma'] * df['moneyness']

# Optional: Drop unhelpful columns if needed
# df.drop(columns=['daily_sentiment_score', 'month'], inplace=True)

print("✅ Engineered new features.")


✅ Engineered new features.


In [3]:
# 🎯 Define the binary classification target again (just to be safe)
df['label'] = (df['lastPrice'] > df['bid']).astype(int)

# ✅ Updated feature columns for model-v2.0
feature_cols = [
    # Core Option Info
    'strike', 'bid', 'ask', 'spot_price', 'moneyness', 'days_to_expiry',

    # Liquidity
    'volume', 'openInterest',

    # Greeks
    'delta', 'gamma', 'theta', 'vega',

    # Volatility
    'impliedVolatility', 'iv_smoothed_5d', 'hist_vol_20d',

    # Engineered Features
    'mid_price',
    'bs_pct_diff', 'mc_pct_diff', 'norm_spread',
    'theta_days', 'delta_spot', 'gamma_moneyness',

    # Model-based Features
    'bs_diff', 'mc_diff', 'bs_vs_mc'
]

# ❌ Drop weak/unavailable features (like daily_sentiment_score, month)
# If still in the DataFrame from old code, ignore for now

# 🧱 Create feature matrix and target
X = df[feature_cols]
y = df['label']

# ✅ Sanity checks
print(f"✅ Final features shape: {X.shape}")
print("✅ Target distribution:")
print(y.value_counts(normalize=True))


✅ Final features shape: (1869, 25)
✅ Target distribution:
label
1    0.619048
0    0.380952
Name: proportion, dtype: float64


In [4]:
# 🔀 Train-Test Split
from sklearn.model_selection import train_test_split

# We'll use a 75/25 split with stratification to preserve label distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# ✅ Sanity check
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("Target distribution in training set:")
print(y_train.value_counts(normalize=True))
print("Target distribution in test set:")
print(y_test.value_counts(normalize=True))


Training set: 1401 samples
Test set: 468 samples
Target distribution in training set:
label
1    0.618844
0    0.381156
Name: proportion, dtype: float64
Target distribution in test set:
label
1    0.619658
0    0.380342
Name: proportion, dtype: float64


In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np
import pandas as pd

# 🧹 Clean infinities and overly large values
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# 🛠️ Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 📊 Scale the imputed data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# ⚖️ Balance training set using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# ✅ Confirm shapes and balance
print(f"✅ Resampled training shape: {X_train_resampled.shape}")
print("✅ Class balance:")
print(pd.Series(y_train_resampled).value_counts(normalize=True))


✅ Resampled training shape: (1734, 25)
✅ Class balance:
label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [6]:
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV

# ⚙️ Train LightGBM with calibration
lgbm = LGBMClassifier(random_state=42, class_weight='balanced')
calibrated_clf = CalibratedClassifierCV(lgbm, method='sigmoid', cv=5)
calibrated_clf.fit(X_train_resampled, y_train_resampled)


[LightGBM] [Info] Number of positive: 694, number of negative: 693
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5650
[LightGBM] [Info] Number of data points in the train set: 1387, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 694, number of negative: 693
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000725 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5628
[LightGBM] [Info] Number of data points in the train set: 1387, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[Lig

CalibratedClassifierCV(cv=5,
                       estimator=LGBMClassifier(class_weight='balanced',
                                                random_state=42))

In [7]:
from sklearn.metrics import classification_report

y_pred = calibrated_clf.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.87      0.88      0.87       178
           1       0.92      0.92      0.92       290

    accuracy                           0.90       468
   macro avg       0.90      0.90      0.90       468
weighted avg       0.90      0.90      0.90       468



In [9]:
y_proba = calibrated_clf.predict_proba(X_test_scaled)[:, 1]
y_pred_custom = (y_proba >= 0.7).astype(int)
print(classification_report(y_test, y_pred_custom))


              precision    recall  f1-score   support

           0       0.80      0.88      0.84       178
           1       0.92      0.86      0.89       290

    accuracy                           0.87       468
   macro avg       0.86      0.87      0.86       468
weighted avg       0.87      0.87      0.87       468



In [10]:
features = [
    'contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change',
    'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney',
    'contractSize', 'currency', 'ticker', 'expiry', 'days_to_expiry', 'spot_price',
    'moneyness', 'date_collected', 'bs_price', 'mc_price', 'bs_diff', 'mc_diff',
    'bs_vs_mc', 'delta', 'gamma', 'theta', 'iv_smoothed_5d', 'log_return',
    'hist_vol_20d', 'moneyness_class'
]


In [11]:
X = df[features + ['simulated_pnl_expiry']]


In [13]:
y_proba = calibrated_clf.predict_proba(X_test_scaled)[:, 1]  # Use `clf` if not using calibrated_clf

# Copy and enrich X_test
X_test_with_proba = X_test.copy()
X_test_with_proba['predicted_proba'] = y_proba
X_test_with_proba['label'] = y_test.values


In [17]:
# Attach simulated_pnl_expiry from df using index
X_test_with_proba['simulated_pnl_expiry'] = df.loc[X_test_with_proba.index, 'simulated_pnl_expiry']


In [18]:
conf_threshold = 0.7
high_conf_trades = X_test_with_proba[X_test_with_proba['predicted_proba'] >= conf_threshold]


In [22]:
# Only keep trades where model confidence (predicted probability) is high
high_conf_trades = X_test_with_proba[X_test_with_proba['predicted_proba'] > 0.7]

# Pull PnL values from original DataFrame
high_conf_trades['pnl'] = df.loc[high_conf_trades.index, 'simulated_pnl_expiry']

# Recalculate metrics
avg_pnl = high_conf_trades['pnl'].mean()
win_rate = (high_conf_trades['pnl'] > 0).mean()
total_trades = len(high_conf_trades)

print(f"✅ High-Confidence Trade Summary (p > 0.7):")
print(f"Avg PnL per trade: {avg_pnl:.2f}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Total Trades: {total_trades}")


✅ High-Confidence Trade Summary (p > 0.7):
Avg PnL per trade: 0.24
Win Rate: 50.55%
Total Trades: 271


In [23]:
import numpy as np

# Extract returns from high-confidence trades
returns = high_conf_trades['pnl'].values

# Set annual risk-free rate (you can adjust this if needed)
risk_free_rate = 0.01  # 1% annual -> ~0.00004 daily
daily_rf = risk_free_rate / 252  # Adjust based on trading frequency

# ✅ Sharpe Ratio
sharpe_ratio = (np.mean(returns) - daily_rf) / np.std(returns, ddof=1)

# ✅ Sortino Ratio
downside_returns = returns[returns < 0]
downside_std = np.std(downside_returns, ddof=1)
sortino_ratio = (np.mean(returns) - daily_rf) / downside_std if downside_std != 0 else np.nan

# ✅ Calmar Ratio
cumulative = np.cumsum(returns)
running_max = np.maximum.accumulate(cumulative)
drawdown = running_max - cumulative
max_drawdown = np.max(drawdown)
calmar_ratio = np.mean(returns) / max_drawdown if max_drawdown != 0 else np.nan

# ✅ CVaR at 95%
alpha = 0.05
sorted_returns = np.sort(returns)
var_95 = np.percentile(sorted_returns, 100 * alpha)
cvar_95 = sorted_returns[sorted_returns <= var_95].mean()

# ✅ Omega Ratio
threshold = 0  # break-even
gain = returns[returns > threshold].sum()
loss = abs(returns[returns <= threshold].sum())
omega_ratio = gain / loss if loss != 0 else np.inf

# ✅ Print all metrics
print("\n📊 Risk-Adjusted Metrics:")
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Sortino Ratio: {sortino_ratio:.4f}")
print(f"Calmar Ratio: {calmar_ratio:.4f}")
print(f"CVaR (95%): {cvar_95:.4f}")
print(f"Omega Ratio: {omega_ratio:.4f}")



📊 Risk-Adjusted Metrics:
Sharpe Ratio: 0.2150
Sortino Ratio: 0.3364
Calmar Ratio: 0.0356
CVaR (95%): -2.1459
Omega Ratio: 1.9116
